# GoalScope goal-head eval — faithfulness vs causality

Grouped per-anchor bars for the matched pair (`--strategy additive --alpha 0.2 --heads goal`,
n_specs=512, seed=123):

* **faithfulness** (solid) — is the NL read-out a faithful description of the implementation span?
* **causality** (striped) — steered into an information-less C++ stub, does the continuation
  implement the same *idea*? (cross-language transfer)

Both metrics are `hard_yes` (lenient oracle YES-rate). Colors follow `clock/12.1`.

In [ ]:
import sys
sys.path.append("../..")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from scripts.evaluation import faithfulness_io as fio
from src.utils import env_utils

RES = env_utils.DEFAULT_RESULTS_DIR
ANCHOR_ORDER = ["prev_end", "indent", "var", "eq", "first_impl"]
ANCHOR_COLOR = {                       # matte / muted tones (from clock/12.1)
    "prev_end": "#b5675f", "indent": "#6b6aa1", "var": "#d1a054",
    "eq": "#a7c48a", "first_impl": "#3f8079", "OVERALL": "#9a9a9a",
}

# the matched pair: same config, differing only in --mode
LEAF = ("heads=goal__layers=35-46__nsteer=all__maxnew=64__span=rule"
        "__dtype=int__nspec=512__nc=3-4-5__seed=123")
RUN = f"additive/0.2/{LEAF}"

def load(mode):
    d = fio.faithfulness_root(RES, mode) / RUN
    return fio.load_config(d), fio.load_summary(d)

cfg_f, sum_f = load("faithfulness")
cfg_c, sum_c = load("causality")
for name, cfg, s in [("faithfulness", cfg_f, sum_f), ("causality", cfg_c, sum_c)]:
    print(f"{name:13s} strategy={cfg['strategy']} alpha={cfg['alpha']} heads={cfg['heads_name']} "
          f"n_specs={cfg['n_specs']} | n_records={s['n_records']} overall={s['overall']['hard_yes']:.3f}")

In [ ]:
# ── grouped bars: faithfulness (solid) vs causality (striped) ──
anchors = [a for a in ANCHOR_ORDER if a in sum_f["by_anchor"] and a in sum_c["by_anchor"]]
labels = anchors + ["OVERALL"]

faith = [sum_f["by_anchor"][a]["hard_yes"] for a in anchors] + [sum_f["overall"]["hard_yes"]]
causal = [sum_c["by_anchor"][a]["hard_yes"] for a in anchors] + [sum_c["overall"]["hard_yes"]]
ns = [sum_f["by_anchor"][a]["n"] for a in anchors] + [sum_f["n_records"]]
colors = [ANCHOR_COLOR.get(a, "#9a9a9a") for a in labels]

x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(1.55 * len(labels) + 1.5, 4.4))

b1 = ax.bar(x - w / 2, faith, w, color=colors, edgecolor="white", linewidth=0.6)
b2 = ax.bar(x + w / 2, causal, w, color=colors, edgecolor="white", linewidth=0.6,
            hatch="///")                                  # striped = causality
for bars in (b1, b2):
    for r in bars:
        ax.annotate(f"{r.get_height():.2f}", (r.get_x() + r.get_width() / 2, r.get_height()),
                    ha="center", va="bottom", fontsize=8)

ax.axvline(len(anchors) - 0.5, color="grey", ls=":", lw=1)   # divider before OVERALL
ax.set_xticks(x); ax.set_xticklabels([f"{a}\n(n={n})" for a, n in zip(labels, ns)])
ax.set_ylim(0, 1.10); ax.set_ylabel("hard YES-rate (lenient oracle)")
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
ax.set_title(f"GoalScope goal heads · {cfg_f['strategy']} \u03b1={cfg_f['alpha']} · "
             f"{cfg_f['n_heads']} heads · n={cfg_f['n_specs']} specs", fontsize=10, pad=26)
# metric legend: single row, centred just under the title
ax.legend(handles=[Patch(facecolor="#9a9a9a", label="faithfulness (read-out vs impl)"),
                   Patch(facecolor="#9a9a9a", hatch="///", label="causality (C++ transfer)")],
          loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=8,
          frameon=False, borderaxespad=0.0, columnspacing=1.6, handlelength=1.8)

fig.tight_layout()
fig.savefig("goal_faithfulness_vs_causality.pdf", bbox_inches="tight")
fig.savefig("goal_faithfulness_vs_causality.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved goal_faithfulness_vs_causality.{pdf,png}")